# Ricostruzione dello storico dal punto di vista delle istituzioni

In [1]:
import json
import pandas as pd
import numpy as np

from collections import defaultdict

In [2]:
with open('storico_pulito.json', 'r') as outfile:
    storico_pulito = json.load(outfile)

a. Mettere le istituzioni come chiave di un nuovo dizionario
- Creare un nuovo dizionario
- Scorrere autori, anni in storico_pulito
- Recuperare le informazioni all'interno di anni
- Per ogni informazione nella lista, aggiungere una nuova chiave (per ora vuota) al nuovo dizionario

In [3]:
'''def new_dict(old_dict):
    latest_dict = {}

    for autore, anno in old_dict.items():

        for anno, affiliazioni in anno.items():

            for i in affiliazioni:

                nome = i['nome_istituzione']

                if nome not in latest_dict.keys():
                    latest_dict[nome] = anno


    return latest_dict'''

"def new_dict(old_dict):\n    latest_dict = {}\n\n    for autore, anno in old_dict.items():\n\n        for anno, affiliazioni in anno.items():\n\n            for i in affiliazioni:\n\n                nome = i['nome_istituzione']\n\n                if nome not in latest_dict.keys():\n                    latest_dict[nome] = anno\n\n\n    return latest_dict"

In [4]:
# Normalizziamo il dizionario
def recupera_id_mancanti(old_dict):
    '''
    Propaga gli ID mancanti usando altre occorrenze
    della stessa istituzione nel dataset.
    '''

    # ---- PASSO 1: costruzione mappa nome -> id ----
    mappa_id = {}

    for autore, anni in old_dict.items():
        for anno, affiliazioni in anni.items():

            if not isinstance(affiliazioni, list):
                continue

            for i in affiliazioni:
                if isinstance(i, dict):
                    nome = i.get('nome_istituzione')
                    id_ = i.get('id')

                    if nome and id_:
                        mappa_id[nome] = id_

    # ---- PASSO 2: normalizzazione + riempimento ----
    for autore, anni in old_dict.items():
        for anno, affiliazioni in anni.items():

            nuove_aff = []

            if not isinstance(affiliazioni, list):
                continue

            for i in affiliazioni:

                # Caso stringa
                if isinstance(i, str):
                    nome = i
                    id_ = mappa_id.get(nome)
                    nuove_aff.append({
                        'nome_istituzione': nome,
                        'id': id_
                    })

                # Caso dict
                elif isinstance(i, dict):
                    nome = i.get('nome_istituzione')
                    id_ = i.get('id')

                    if nome and not id_:
                        i['id'] = mappa_id.get(nome)

                    nuove_aff.append(i)

            old_dict[autore][anno] = nuove_aff

    return old_dict

In [5]:
storico_pulito = recupera_id_mancanti(storico_pulito)

In [6]:
def new_dict(old_dict):

    '''
    Funzione per creare un nuovo dizionario che abbia come chiave le istituzioni e come valori i dipendenti per ogni anno.
    '''

    latest_dict = {}

    for autore, anni in old_dict.items():

        for anno, affiliazioni in anni.items():

            for i in affiliazioni:
                
                nome = i['nome_istituzione']

                if nome not in latest_dict:
                    latest_dict[nome] = {}

                if anno not in latest_dict[nome]:
                    latest_dict[nome][anno] = []

                if autore not in latest_dict[nome][anno]:
                    latest_dict[nome][anno].append(autore)

    return latest_dict

In [7]:
latest_dict = new_dict(storico_pulito)

In [8]:
def delete_previous_years(dizionario):

    for istituzione, informazioni in list(dizionario.items()):
        for anno, studiosi in list(informazioni.items()):
            if int(anno) < 2019:
                del dizionario[istituzione][anno]


    for istituzione in dizionario:
                dizionario[istituzione] = dict(sorted(dizionario[istituzione].items()))
    return dizionario

In [9]:
#Versione finale del dizionario con solamente gli anni dal 2020 al 2024
almost_there = delete_previous_years(latest_dict)

In [10]:
almost_there

{'Mario Negri Institute for Pharmacological Research': {'2019': ['Valter Torri',
   'Michela Cinquini',
   'Luca Clivio',
   'Ettore Beghi',
   'Annamaria Vezzani',
   'Chiara Stival',
   'Annalisa Perna',
   'Elisa R. Zanier',
   'Gianluigi Forloni',
   'Laura Reale',
   'Massimo Cartabia',
   'Michele Zanetti',
   'Maurizio Bonati',
   'Anna Caroli',
   'Silvano Gallus',
   'Federica Frigerio',
   'Silvio Garattini',
   'Ugo Lucca',
   'Alberto Pauletti',
   'Alessia Salamone',
   'Massimo Rizzi',
   'Rosaria Pascente',
   'Bianca R. Villa',
   'Gaetano Terrone',
   'Claudia Balducci',
   'Andrea Colombo',
   'Giovanni Nattino',
   'Guido Bertolini',
   'Alessandro Delitala',
   'Rita Campi',
   'Giulia Segre',
   'Alessandro Nobili',
   'Maria Grazia De Simoni',
   'Paola Mosconi',
   'Filippo Sean Giorgi',
   'Claudia Fracasso',
   'Barbara D’Avanzo',
   'Vittorio Bertelè',
   'Mirjana Carli',
   'Marta Baviera',
   'Roberto William Invernizzi',
   'Sara Castiglioni',
   'Orso Bugi

In [11]:
len(almost_there)

1488

In [12]:
vuote = [inst for inst, bilanci in almost_there.items() if not bilanci]

len(vuote)

166

Adesso trasformiamo i nomi in bilanci-> calcoliamo il numero di persone entrate, il numero di persone uscite e il numero di persone presenti per ogni anno

In [13]:
def bilancio(dizionario):
    
    new_dict = {}

    # questo controllo elimina le istituzioni che non hanno anni
    for istituzione, bilanci in dizionario.items():
        
        for anno, affiliazioni in bilanci.items():
            tot_affiliazioni_per_anno = len(affiliazioni)

            # prendiamo gli anni precedente e successivo a quello che stiamo considerando
            anno_precedente = str(int(anno)-1)
            anno_successivo = str(int(anno)+1) #devono essere ritrasformati in stringa sennò non si può fare confronto dopo

            # creiamo l'entrata nel nuovo dizionario
            if istituzione not in new_dict:
                new_dict[istituzione] = {}

            # se l'anno che stiamo controllando non si trova già associato all'istituzione, creiamo la chiave con sottodizionario
            if anno not in new_dict[istituzione]:
                new_dict[istituzione][anno] = {'tot_dipendenti_per_anno': tot_affiliazioni_per_anno, 'studiosi_usciti': 0, 'studiosi_entrati': 0}

            # per ogni studioso nelle affiliazioni
            if anno_precedente in bilanci:
                studiosi_precedenti = bilanci[anno_precedente]
                for studioso in affiliazioni:
                    #se non era presente l'anno prima, si aggiunge uno al conto degli studiosi entrati
                    if studioso not in studiosi_precedenti:
                        new_dict[istituzione][anno]['studiosi_entrati'] += 1

            # stesso discorso ma per studiosi usciti in riferimento all'anno successivo
            if anno_successivo in bilanci:
                studiosi_successivi = bilanci[anno_successivo]
                for studioso in affiliazioni:
                    if studioso not in studiosi_successivi:
                        new_dict[istituzione][anno]['studiosi_usciti'] += 1
            

    for inst, balance in list(new_dict.items()):
        for year, affs in list(balance.items()):
            if int(year) < 2019:
                del new_dict[inst][year]

    return new_dict


In [14]:
version_3_dict = bilancio(almost_there)

In [15]:
version_3_dict

{'Mario Negri Institute for Pharmacological Research': {'2019': {'tot_dipendenti_per_anno': 206,
   'studiosi_usciti': 25,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 210,
   'studiosi_usciti': 39,
   'studiosi_entrati': 29},
  '2021': {'tot_dipendenti_per_anno': 182,
   'studiosi_usciti': 36,
   'studiosi_entrati': 11},
  '2022': {'tot_dipendenti_per_anno': 197,
   'studiosi_usciti': 60,
   'studiosi_entrati': 51},
  '2023': {'tot_dipendenti_per_anno': 164,
   'studiosi_usciti': 32,
   'studiosi_entrati': 27},
  '2024': {'tot_dipendenti_per_anno': 158,
   'studiosi_usciti': 36,
   'studiosi_entrati': 26},
  '2025': {'tot_dipendenti_per_anno': 130,
   'studiosi_usciti': 0,
   'studiosi_entrati': 8}},
 'Azienda Socio Sanitaria Territoriale degli Spedali Civili di Brescia': {'2019': {'tot_dipendenti_per_anno': 180,
   'studiosi_usciti': 47,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 202,
   'studiosi_usciti': 52,
   'studiosi_entrati': 69},
  '2

In [16]:
# Creiamo il dizionario contenente solo le info del 2025
def make_dict_2025(dictionary): 
    dict_2025 = {}
    for istituzione, informazioni in dictionary.items():
        for anno, bilanci in informazioni.items():
            if anno == '2025':
                if istituzione not in dict_2025:
                    dict_2025[istituzione] = {}
                dict_2025[istituzione][anno] = bilanci

    return dict_2025

In [17]:
dict_2025 = make_dict_2025(version_3_dict)

In [18]:
def delete_2025(dictionary):
    for inst, balance in list(dictionary.items()):
        for year, affs in list(balance.items()):
            if year == 'id' or year == 'openalex_id':
                continue
            if int(year) == 2025:
                del dictionary[inst][year]

    return dictionary

version_3_dict = delete_2025(version_3_dict)

In [19]:
version_3_dict

{'Mario Negri Institute for Pharmacological Research': {'2019': {'tot_dipendenti_per_anno': 206,
   'studiosi_usciti': 25,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 210,
   'studiosi_usciti': 39,
   'studiosi_entrati': 29},
  '2021': {'tot_dipendenti_per_anno': 182,
   'studiosi_usciti': 36,
   'studiosi_entrati': 11},
  '2022': {'tot_dipendenti_per_anno': 197,
   'studiosi_usciti': 60,
   'studiosi_entrati': 51},
  '2023': {'tot_dipendenti_per_anno': 164,
   'studiosi_usciti': 32,
   'studiosi_entrati': 27},
  '2024': {'tot_dipendenti_per_anno': 158,
   'studiosi_usciti': 36,
   'studiosi_entrati': 26}},
 'Azienda Socio Sanitaria Territoriale degli Spedali Civili di Brescia': {'2019': {'tot_dipendenti_per_anno': 180,
   'studiosi_usciti': 47,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 202,
   'studiosi_usciti': 52,
   'studiosi_entrati': 69},
  '2021': {'tot_dipendenti_per_anno': 212,
   'studiosi_usciti': 63,
   'studiosi_entrati': 62},
  

In [20]:
storico_pulito

{'Valter Torri': {'1999': [{'nome_istituzione': 'Mario Negri Institute for Pharmacological Research',
    'id': 'https://openalex.org/I19630809'},
   {'nome_istituzione': 'Azienda Socio Sanitaria Territoriale degli Spedali Civili di Brescia',
    'id': 'https://openalex.org/I4210103289'},
   {'nome_istituzione': 'Ospedale Infermi di Rimini',
    'id': 'https://openalex.org/I4210129405'},
   {'nome_istituzione': 'University of Brescia',
    'id': 'https://openalex.org/I79940851'}],
  '2000': [{'nome_istituzione': 'Mario Negri Institute for Pharmacological Research',
    'id': 'https://openalex.org/I19630809'}],
  '2001': [{'nome_istituzione': 'Mario Negri Institute for Pharmacological Research',
    'id': 'https://openalex.org/I19630809'},
   {'nome_istituzione': 'Azienda Socio Sanitaria Territoriale degli Spedali Civili di Brescia',
    'id': 'https://openalex.org/I4210103289'},
   {'nome_istituzione': 'Ospedale Infermi di Rimini',
    'id': 'https://openalex.org/I4210129405'},
   {'no

In [21]:
def add_id(dictionary, latest_dictionary):

    id_per_istituzione = {}

    for studioso, anni in dictionary.items():
        for anno, istituzioni in anni.items():
            for entry in istituzioni:
                nome = entry['nome_istituzione']
                inst_id = entry['id']

                # salva solo la prima occorrenza
                if nome not in id_per_istituzione:
                    id_per_istituzione[nome] = inst_id
    
    for istituzione, anni in latest_dictionary.items():
        istituzione_id = id_per_istituzione.get(istituzione)
        if istituzione_id:
            anni['id'] = istituzione_id




    return latest_dictionary

In [22]:
final_version = add_id(storico_pulito, version_3_dict)

In [23]:
final_version

{'Mario Negri Institute for Pharmacological Research': {'2019': {'tot_dipendenti_per_anno': 206,
   'studiosi_usciti': 25,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 210,
   'studiosi_usciti': 39,
   'studiosi_entrati': 29},
  '2021': {'tot_dipendenti_per_anno': 182,
   'studiosi_usciti': 36,
   'studiosi_entrati': 11},
  '2022': {'tot_dipendenti_per_anno': 197,
   'studiosi_usciti': 60,
   'studiosi_entrati': 51},
  '2023': {'tot_dipendenti_per_anno': 164,
   'studiosi_usciti': 32,
   'studiosi_entrati': 27},
  '2024': {'tot_dipendenti_per_anno': 158,
   'studiosi_usciti': 36,
   'studiosi_entrati': 26},
  'id': 'https://openalex.org/I19630809'},
 'Azienda Socio Sanitaria Territoriale degli Spedali Civili di Brescia': {'2019': {'tot_dipendenti_per_anno': 180,
   'studiosi_usciti': 47,
   'studiosi_entrati': 0},
  '2020': {'tot_dipendenti_per_anno': 202,
   'studiosi_usciti': 52,
   'studiosi_entrati': 69},
  '2021': {'tot_dipendenti_per_anno': 212,
   'studiosi_u

In [24]:
with open('balance_data.json', 'w') as f:
    json.dump(final_version, f, indent=2)

In [25]:
#with open('dict_2025.json', 'w') as f:
#    json.dump(dict_2025, f, indent=2)